# Reproducing and extending Table 2.1: AR-order selection

This notebook compares **BIC**, the **PACF cutoff**, and eight shared-breakpoint L-selectors on the same 50 simulated AR($p$) processes for each $p\in\{1,\ldots,6\}$. The AR(1)–AR(3) columns reproduce Table 2.1; AR(4)–AR(6) extend the experiment without changing its design. Each process has $T=2{,}000$ retained observations after a burn-in of 1,000, $\operatorname{Uniform}[-1,1]$ innovations, and empirical minimum widths $\widehat m_k$ for $k=1,\ldots,10$.

The eight L-selectors are the full $2\times2\times2$ combination of:

1. **tail model:** affine or constant (the head is affine in both cases);
2. **loss:** squared or absolute;
3. **score:** segment-size weighted or total loss.

For a candidate order $q$, both fits include the observed point at $k=q+1$: the head uses $k=1,\ldots,q+1$ and the tail uses $k=q+1,\ldots,K$. Thus every L-selector can select AR(1). The two segment fits remain independent; sharing the observation does not constrain their fitted values to agree there. The weighted construction adapts the weighting principle in [Salvador and Chan (2004), Section 3.2](https://cs.fit.edu/~pkc/papers/ictai04salvador.pdf) to the shared-breakpoint setting. The exact implementations are in [`ar_tools/L_method.py`](ar_tools/L_method.py) and [`ar_tools/classical_selectors.py`](ar_tools/classical_selectors.py).


In [5]:
import hashlib
import itertools                                                                                     # infrastructure
import json                                                                                        # infrastructure
import sys                                                                                         # infrastructure
from pathlib import Path                                                                           # infrastructure
from time import perf_counter

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from threadpoolctl import threadpool_limits                                                        # infrastructure

AR_CODE_DIR = next((                                                                               # infrastructure
    candidate.resolve()                                                                            # infrastructure
    for root in (Path.cwd(), *Path.cwd().parents)                                                  # infrastructure
    for candidate in (root / "Code" / "Code_AR_processes", root)                                 # infrastructure
    if (candidate / "ar_tools/functions.py").is_file()                                             # infrastructure
    and (candidate / "ar_tools/L_method.py").is_file()                                             # infrastructure
), None)                                                                                           # infrastructure
if AR_CODE_DIR is None:                                                                            # infrastructure
    raise FileNotFoundError("Could not locate Code/Code_AR_processes.")                             # infrastructure
PROJECT_ROOT = AR_CODE_DIR.parents[1]                                                              # infrastructure
if str(PROJECT_ROOT) not in sys.path:                                                              # infrastructure
    sys.path.insert(0, str(PROJECT_ROOT))                                                          # infrastructure

from Code.Code_AR_processes.ar_tools.classical_selectors import (
    aic_bic_ar_order,
    pacf_cutoff_order,
)
from Code.Code_AR_processes.ar_tools.descent import batched_geometric_width_curves
from Code.Code_AR_processes.ar_tools.functions import (
    ar_impulse_response,
    companion_matrix,
    jones_ar_coefficients,
    simulate_uniform_ar,
)
from Code.Code_AR_processes.ar_tools.L_method import l_method_selector

NOTEBOOK_NAME = "ar_order_minimum_width"                                                          # infrastructure
RESULTS_DIR = AR_CODE_DIR / "outputs" / "csv" / NOTEBOOK_NAME                                    # infrastructure
RESULTS_DIR.mkdir(parents=True, exist_ok=True)                                                     # infrastructure

true_orders = tuple(range(1, 7))
settings = {
    "true_orders": list(true_orders),
    "replications_per_order": 50,
    "sample_size": 2_000,
    "burnin": 1_000,
    "half_width": 1.0,
    "k_values": list(range(1, 11)),
    "seeds": {
        "1": 20260813, "2": 20260814, "3": 20260816,
        "4": 20260817, "5": 20260818, "6": 20260819,
    },
    "starts": 3,
    "iterations": 500,
    "initial_step": 0.2,
    "geometric_decay": 0.99,
    "width_source_hashes": {                                                                      # infrastructure
        name: hashlib.sha256((AR_CODE_DIR / name).read_bytes()).hexdigest()                         # infrastructure
        for name in ("ar_tools/functions.py", "ar_tools/descent.py")                              # infrastructure
    },                                                                                             # infrastructure
}
k_values = np.asarray(settings["k_values"])

l_method_specs = []
for tail_label, affine_affine in (("affine", True), ("constant", False)):
    for loss_label, squared_loss in (("squared", True), ("absolute", False)):
        for score_label, weighted in (("weighted", True), ("total", False)):
            if squared_loss:
                metric_label = "weighted RMSE" if weighted else "total SSE"
            else:
                metric_label = "weighted MAE" if weighted else "total SAE"
            l_method_specs.append({
                "method": f"L — {tail_label} tail — {loss_label} — {metric_label}",
                "tail": tail_label,
                "loss": loss_label,
                "score": score_label,
                "affine_affine": affine_affine,
                "squared_loss": squared_loss,
                "weighted": weighted,
            })

method_order = ["BIC", "PACF cutoff"] + [spec["method"] for spec in l_method_specs]
print("50 processes per true order; p=1,...,6; T=2,000; K=10; ten selectors on matched data.")


50 processes per true order; p=1,...,6; T=2,000; K=10; ten selectors on matched data.


## Paired simulation

One deterministic random-number stream generates the coefficients and time series for each replication. All ten selectors therefore see matched data. BIC and PACF consider orders $0,\ldots,9$; every L-selector sees the same empirical width curve. A validated cache preserves the original AR(1)–AR(3) curves and computes only missing orders or replications.


In [6]:
configuration_path = RESULTS_DIR / "configuration.json"                                # infrastructure
width_path = RESULTS_DIR / "width_curves.csv"                                                      # infrastructure
fixed_cache_keys = (                                                                               # infrastructure
    "replications_per_order", "sample_size", "burnin", "half_width",                            # infrastructure
    "k_values", "starts", "iterations", "initial_step",                                        # infrastructure
    "geometric_decay", "width_source_hashes",                                                    # infrastructure
)                                                                                                 # infrastructure
cached_settings = (                                                                               # infrastructure
    json.loads(configuration_path.read_text())                                                     # infrastructure
    if configuration_path.is_file() else {}                                                       # infrastructure
)                                                                                                 # infrastructure
cache_compatible = all(                                                                           # infrastructure
    cached_settings.get(key) == settings[key] for key in fixed_cache_keys                         # infrastructure
)                                                                                                 # infrastructure

width_columns = ["true_order", "replication", "k", "minimum_width", "coefficients"]             # infrastructure
if cache_compatible and width_path.is_file():                                                      # infrastructure
    cached_widths = pd.read_csv(width_path)                                                        # infrastructure
    cached_orders = set(cached_widths["true_order"].astype(int).unique())                          # infrastructure
    cache_compatible = all(                                                                        # infrastructure
        cached_settings.get("seeds", {}).get(str(order)) == settings["seeds"][str(order)]          # infrastructure
        for order in cached_orders & set(true_orders)                                              # infrastructure
    )                                                                                              # infrastructure
else:                                                                                              # infrastructure
    cached_widths = pd.DataFrame(columns=width_columns)                                            # infrastructure

valid_groups = []                                                                                  # infrastructure
if cache_compatible:                                                                              # infrastructure
    cached_widths = cached_widths[                                                                 # infrastructure
        cached_widths["true_order"].isin(true_orders)                                             # infrastructure
        & cached_widths["replication"].between(1, settings["replications_per_order"])              # infrastructure
    ]                                                                                              # infrastructure
    required_k = set(k_values.tolist())                                                            # infrastructure
    for _, group in cached_widths.groupby(["true_order", "replication"]):                         # infrastructure
        if (                                                                                       # infrastructure
            len(group) == len(k_values)                                                            # infrastructure
            and set(group["k"].astype(int)) == required_k                                         # infrastructure
            and not group["k"].duplicated().any()                                                 # infrastructure
            and np.isfinite(group["minimum_width"]).all()                                        # infrastructure
        ):                                                                                         # infrastructure
            valid_groups.append(group[width_columns])                                              # infrastructure
width_curves = (                                                                                   # infrastructure
    pd.concat(valid_groups, ignore_index=True)                                                     # infrastructure
    if valid_groups else pd.DataFrame(columns=width_columns)                                       # infrastructure
)                                                                                                 # infrastructure

expected_processes = {                                                                            # infrastructure
    (true_order, replication)                                                                     # infrastructure
    for true_order in true_orders                                                                 # infrastructure
    for replication in range(1, settings["replications_per_order"] + 1)                           # infrastructure
}                                                                                                 # infrastructure
cached_processes = set(map(tuple, width_curves[["true_order", "replication"]].astype(int).drop_duplicates().to_numpy()))  # infrastructure
missing_processes = sorted(expected_processes - cached_processes)                                  # infrastructure

if missing_processes:
    width_rows = width_curves.to_dict("records")
    started = perf_counter()
    with threadpool_limits(limits=1):                                                              # infrastructure
        for true_order in true_orders:
            experiment_seed = settings["seeds"][str(true_order)]
            missing_replications = [
                replication for order, replication in missing_processes
                if order == true_order
            ]
            for completed, replication in enumerate(missing_replications, start=1):
                replication_index = replication - 1
                generator = np.random.default_rng(np.random.SeedSequence([
                    experiment_seed, replication_index, 0,
                ]))
                coefficients = jones_ar_coefficients(true_order, rng=generator)
                series = simulate_uniform_ar(
                    coefficients,
                    sample_size=settings["sample_size"],
                    burnin=settings["burnin"],
                    half_width=settings["half_width"],
                    rng=generator,
                )
                _, minimum_widths, _ = batched_geometric_width_curves(
                    series,
                    int(k_values.max()),
                    seed=experiment_seed,
                    replication=replication_index,
                    starts=settings["starts"],
                    iterations=settings["iterations"],
                    initial_step=settings["initial_step"],
                    geometric_decay=settings["geometric_decay"],
                )
                for k, width in enumerate(minimum_widths, start=1):
                    width_rows.append({
                        "true_order": true_order,
                        "replication": replication,
                        "k": int(k),
                        "minimum_width": float(width),
                        "coefficients": json.dumps(coefficients.tolist()),
                    })
                if completed % 10 == 0:
                    print(
                        f"Completed {completed}/{len(missing_replications)} "
                        f"new AR({true_order}) curves"
                    )
    width_curves = (
        pd.DataFrame(width_rows)[width_columns]
        .sort_values(["true_order", "replication", "k"])
        .reset_index(drop=True)
    )
    print(
        f"Generated {len(missing_processes)} missing width curves "
        f"in {perf_counter() - started:.1f} seconds."
    )
else:
    print(f"Reused all {len(expected_processes)} cached width curves.")

width_curves.to_csv(width_path, index=False)                                                       # infrastructure
configuration_path.write_text(json.dumps(settings, indent=2) + "\n")                             # infrastructure
expected_width_rows = len(expected_processes) * len(k_values)                                     # infrastructure
assert len(width_curves) == expected_width_rows
assert set(map(tuple, width_curves[["true_order", "replication"]].astype(int).drop_duplicates().to_numpy())) == expected_processes


Reused all 300 cached width curves.


In [7]:
selection_rows = []

for (true_order, replication), curve in width_curves.groupby(
    ["true_order", "replication"], sort=True
):
    true_order = int(true_order)
    replication = int(replication)
    experiment_seed = settings["seeds"][str(true_order)]

    # Regenerate the matched process used for this cached width curve.
    generator = np.random.default_rng(np.random.SeedSequence([
        experiment_seed, replication - 1, 0,
    ]))
    coefficients = jones_ar_coefficients(true_order, rng=generator)
    cached_coefficients = np.asarray(json.loads(curve.iloc[0]["coefficients"]))
    assert np.array_equal(coefficients, cached_coefficients)
    series = simulate_uniform_ar(
        coefficients,
        sample_size=settings["sample_size"],
        burnin=settings["burnin"],
        half_width=settings["half_width"],
        rng=generator,
    )

    _, bic_order, _ = aic_bic_ar_order(series, maximum_order=9)
    pacf_order, _, _ = pacf_cutoff_order(series, maximum_order=9)
    selected_orders = {
        "BIC": bic_order,
        "PACF cutoff": pacf_order,
    }

    minimum_widths = curve.sort_values("k")["minimum_width"].to_numpy()
    for spec in l_method_specs:
        fit = l_method_selector(
            minimum_widths,
            weighted=spec["weighted"],
            common_breakpoint=True,
            affine_affine=spec["affine_affine"],
            squared_loss=spec["squared_loss"],
        )
        selected_orders[spec["method"]] = fit["selected_order"]

    for method, selected_order in selected_orders.items():
        selection_rows.append({
            "true_order": true_order,
            "replication": replication,
            "method": method,
            "selected_order": int(selected_order),
        })

selections = pd.DataFrame(selection_rows)
selections.to_csv(RESULTS_DIR / "selections.csv", index=False)                                    # infrastructure

summary = (
    selections.assign(
        correct=lambda frame: frame["selected_order"] == frame["true_order"],
        under=lambda frame: frame["selected_order"] < frame["true_order"],
        over=lambda frame: frame["selected_order"] > frame["true_order"],
    )
    .groupby(["method", "true_order"], as_index=False)[["correct", "under", "over"]]
    .sum()
)
summary[["correct", "under", "over"]] = summary[["correct", "under", "over"]].astype(int)
summary["total"] = summary[["correct", "under", "over"]].sum(axis=1)
summary.to_csv(RESULTS_DIR / "summary.csv", index=False)                                           # infrastructure

assert (summary["total"] == 50).all()
assert len(selections) == len(true_orders) * 50 * len(method_order)

# These are the published BIC and PACF rows in Table 2.1.
table_2_1_reference = {
    ("BIC", 1): (47, 3, 0), ("BIC", 2): (46, 4, 0), ("BIC", 3): (43, 7, 0),
    ("PACF cutoff", 1): (31, 1, 18), ("PACF cutoff", 2): (34, 1, 15),
    ("PACF cutoff", 3): (33, 3, 14),
}
for (method, true_order), expected in table_2_1_reference.items():
    observed = summary.loc[
        (summary["method"] == method) & (summary["true_order"] == true_order),
        ["correct", "under", "over"],
    ].iloc[0]
    assert tuple(observed) == expected

print("Evaluated all ten selectors for AR(1)–AR(6); the original BIC and PACF columns still match Table 2.1.")


Evaluated all ten selectors for AR(1)–AR(6); the original BIC and PACF columns still match Table 2.1.


## Selection table

Each cell reports **correct selections / 50**, followed by **(underselections / overselections)**.


In [8]:
summary_lookup = summary.set_index(["method", "true_order"])

index_rows = [
    ("BIC", "—", "—", "—"),
    ("PACF cutoff", "—", "—", "—"),
]
for spec in l_method_specs:
    index_rows.append((
        "L-method",
        f"{spec['tail']} tail",
        spec["loss"],
        spec["score"],
    ))

table_index = pd.MultiIndex.from_tuples(
    index_rows,
    names=["Selector", "Tail", "Loss", "Score"],
)
comparison_table = pd.DataFrame(index=table_index)
for true_order in true_orders:
    entries = []
    for method in method_order:
        row = summary_lookup.loc[(method, true_order)]
        entries.append(
            f"<strong>{int(row['correct'])}/50</strong><br>"
            f"<span>({int(row['under'])}/{int(row['over'])})</span>"
        )
    comparison_table[f"AR({true_order})"] = entries

comparison_table.to_csv(RESULTS_DIR / "table_2_1_comparison.csv")                                 # infrastructure

styled_table = (
    comparison_table.style
    .set_caption("AR-order selections on 50 matched processes for each p=1,...,6")
    .set_properties(**{
        "text-align": "center",
        "min-width": "78px",
        "padding": "8px 10px",
        "font-size": "14px",
    })
    .set_table_styles([
        {"selector": "table", "props": [
            ("border-collapse", "collapse"),
            ("font-family", "-apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif"),
            ("border", "1px solid #c7d0dc"),
        ]},
        {"selector": "caption", "props": [
            ("caption-side", "top"),
            ("font-size", "17px"),
            ("font-weight", "650"),
            ("color", "#17324d"),
            ("padding", "0 0 10px 0"),
            ("text-align", "left"),
        ]},
        {"selector": "thead th", "props": [
            ("background-color", "#17324d"),
            ("color", "white"),
            ("font-weight", "600"),
            ("padding", "8px 12px"),
            ("border", "1px solid #294b6d"),
        ]},
        {"selector": "tbody th", "props": [
            ("text-align", "left"),
            ("padding", "8px 10px"),
            ("background-color", "#f3f6f9"),
            ("border", "1px solid #d8dee7"),
            ("font-weight", "500"),
        ]},
        {"selector": "tbody td", "props": [
            ("border", "1px solid #d8dee7"),
            ("background-color", "white"),
        ]},
        {"selector": "tbody td span", "props": [
            ("color", "#5d6875"),
            ("font-size", "12px"),
        ]},
        {"selector": "tbody tr:nth-child(3) th, tbody tr:nth-child(3) td", "props": [
            ("border-top", "3px solid #7d94aa"),
        ]},
        {"selector": "tbody tr:nth-child(7) th, tbody tr:nth-child(7) td", "props": [
            ("border-top", "2px solid #aebdca"),
        ]},
        {"selector": "tbody tr:hover td", "props": [
            ("background-color", "#eef5fb"),
        ]},
    ])
)
styled_table


The two absolute-loss score columns agree within each tail family in this experiment. This is required by the implementation: with a shared breakpoint, the weighted MAE is the total absolute error divided by the fixed combined segment size $K+1$, so the two scores have the same minimizer; see [`l_method_selector`](ar_tools/L_method.py). They remain listed separately to display the requested $2\times2\times2$ design explicitly.


## Diagnostic using population minimum widths

The empirical comparison mixes three effects: the L-selector, finite-sample point-cloud error, and numerical minimization of the empirical width. To isolate the selector, this section replaces every empirical curve by the corresponding population minimum-width curve.

For an AR($p$) coefficient vector $\phi$, impulse response $\psi$, and unit direction $u\in\mathbb S^{k-1}$, the population width is
$$
w_k(u)=2\sum_{j=0}^{\infty}\left|\sum_{r=0}^{k-1}u_r\psi_{j-r}\right|,
\qquad m_k=\min_{\lVert u\rVert_2=1}w_k(u).
$$
For $k\ge p+1$, the exact value is
$$
m_k=\frac{2}{\sqrt{1+\phi_1^2+\cdots+\phi_p^2}}.
$$
For $k\le p$, the code below approximates the sphere minimum numerically. These formulas and the strict decrease before the plateau are established in [Proposition `prop:minimum_width` of the thesis source](../../Abschlussarbeit_2021/LaTeX/chapters/3_application_to_time_series.tex); the shared population-width evaluation is implemented in [`ar_tools/functions.py`](ar_tools/functions.py).

The approximation uses pole-factor, coordinate, nested-dimension, and random starts; smooth continuation on the sphere; and finite-zonotope facet-normal refinement. The impulse-response horizon adapts to the spectral radius and is chosen between 20,000 and 250,000 so that $\rho^H\le10^{-10}$ whenever the cap is not active. The table is therefore an approximation only for $k\le p$; the plateau is exact.


In [9]:
def _population_generator_matrix(impulse_response, dimension):
    """Rows are g_j=(psi_j, psi_{j-1}, ..., psi_{j-k+1})."""
    padded = np.r_[np.zeros(dimension - 1), impulse_response]
    return np.column_stack([
        padded[dimension - 1 - lag:dimension - 1 - lag + len(impulse_response)]
        for lag in range(dimension)
    ])


def _normalized_rows(vectors):
    vectors = np.asarray(vectors, dtype=float)
    norms = np.linalg.norm(vectors, axis=1)
    return vectors[norms > 1e-12] / norms[norms > 1e-12, None]


def _pole_factor_starts(coefficients, dimension):
    if dimension == 1:
        return []
    poles = np.linalg.eigvals(companion_matrix(coefficients))
    starts = []
    for indices in itertools.combinations(range(len(coefficients)), dimension - 1):
        factor = np.poly(poles[list(indices)])
        if np.max(np.abs(factor.imag)) < 1e-8:
            starts.append(factor.real / np.linalg.norm(factor.real))
    return starts


def _smooth_population_refinement(generator_matrix, start):
    direction = np.asarray(start, dtype=float)
    direction /= np.linalg.norm(direction)
    scale = max(float(np.max(np.linalg.norm(generator_matrix, axis=1))), 1.0)

    for epsilon_factor in (1e-3, 1e-5, 1e-7):
        epsilon = epsilon_factor * scale

        def objective_and_gradient(vector):
            vector_norm = np.linalg.norm(vector)
            unit_vector = vector / vector_norm
            projections = generator_matrix @ unit_vector
            smoothed_absolute = np.sqrt(projections**2 + epsilon**2)
            objective = 2.0 * np.sum(smoothed_absolute - epsilon)
            sphere_gradient = 2.0 * (
                generator_matrix.T @ (projections / smoothed_absolute)
            )
            gradient = (
                sphere_gradient
                - unit_vector * (unit_vector @ sphere_gradient)
            ) / vector_norm
            return objective, gradient

        result = minimize(
            objective_and_gradient,
            direction,
            method="L-BFGS-B",
            jac=True,
            options={
                "maxiter": 120, "ftol": 1e-13,
                "gtol": 1e-9, "maxls": 40,
            },
        )
        direction = result.x / np.linalg.norm(result.x)
    return direction


def _facet_normal_refinement(generator_matrix, directions, nearest=12):
    row_norms = np.linalg.norm(generator_matrix, axis=1)
    active = row_norms > 1e-11 * row_norms.max()
    rows = generator_matrix[active]
    unit_rows = rows / row_norms[active, None]
    candidates = [direction / np.linalg.norm(direction) for direction in directions]

    for direction in directions:
        distances = np.abs(unit_rows @ direction)
        count = min(nearest, len(rows))
        nearest_rows = rows[np.argpartition(distances, count - 1)[:count]]
        for indices in itertools.combinations(
            range(len(nearest_rows)), generator_matrix.shape[1] - 1
        ):
            _, singular_values, right_vectors = np.linalg.svd(
                nearest_rows[list(indices)], full_matrices=True
            )
            if (
                len(singular_values)
                and singular_values[-1] < 1e-10 * singular_values[0]
            ):
                continue
            normal = right_vectors[-1]
            normal_norm = np.linalg.norm(normal)
            if normal_norm > 1e-12:
                candidates.append(normal / normal_norm)

    candidate_matrix = np.asarray(candidates)
    widths = 2.0 * np.sum(
        np.abs(generator_matrix @ candidate_matrix.T), axis=0
    )
    best_indices = np.argsort(widths)[:8]
    return [candidate_matrix[index] for index in best_indices]


def approximate_population_width_curve(coefficients, seed):
    """Approximate m_k for k<=p and use the exact plateau for k>=p+1."""
    coefficients = np.asarray(coefficients, dtype=float)
    order = len(coefficients)
    spectral_radius = float(max(abs(np.linalg.eigvals(companion_matrix(coefficients)))))
    evaluation_history = min(
        250_000,
        max(20_000, int(np.ceil(np.log(1e-10) / np.log(spectral_radius)))),
    )
    optimization_history = min(3_000, evaluation_history)
    evaluation_impulse = ar_impulse_response(coefficients, evaluation_history)
    optimization_impulse = evaluation_impulse[:optimization_history + 1]
    plateau = 2.0 / np.linalg.norm(np.r_[1.0, -coefficients])
    generator = np.random.default_rng(seed)

    widths = []
    directions = []
    half_history_increments = []
    previous_direction = None

    for dimension in range(1, order + 1):
        optimization_generators = _population_generator_matrix(
            optimization_impulse, dimension
        )
        evaluation_generators = _population_generator_matrix(
            evaluation_impulse, dimension
        )

        starts = []
        if previous_direction is not None:
            starts.append(np.r_[previous_direction, 0.0])
        starts.extend(np.eye(dimension))
        starts.append(np.r_[1.0, -coefficients[:dimension - 1]])
        starts.extend(_pole_factor_starts(coefficients, dimension))
        starts.extend(_normalized_rows(generator.normal(size=(48, dimension))))
        starts = _normalized_rows(starts)

        start_widths = 2.0 * np.sum(
            np.abs(optimization_generators @ starts.T), axis=0
        )
        best_starts = starts[np.argsort(start_widths)[:8]]
        smooth_directions = [
            _smooth_population_refinement(optimization_generators, start)
            for start in best_starts
        ]
        candidate_directions = _facet_normal_refinement(
            optimization_generators, smooth_directions, nearest=12
        ) + smooth_directions
        candidate_matrix = np.asarray(candidate_directions)
        evaluation_widths = 2.0 * np.sum(
            np.abs(evaluation_generators @ candidate_matrix.T), axis=0
        )
        best_index = int(np.argmin(evaluation_widths))
        direction = candidate_matrix[best_index]
        width = float(evaluation_widths[best_index])
        half_width = float(2.0 * np.sum(np.abs(
            evaluation_generators[:len(evaluation_generators) // 2] @ direction
        )))

        widths.append(width)
        directions.append(direction)
        half_history_increments.append(width - half_width)
        previous_direction = direction

    widths.extend([plateau] * (len(k_values) - order))
    return {
        "widths": np.asarray(widths),
        "directions": directions,
        "half_history_increments": np.asarray(half_history_increments),
        "evaluation_history": evaluation_history,
        "spectral_radius": spectral_radius,
        "plateau": plateau,
    }


In [10]:
population_width_path = RESULTS_DIR / "population_width_curves.csv"                     # infrastructure
population_configuration_path = (                                                               # infrastructure
    RESULTS_DIR / "population_configuration.json"                                                # infrastructure
)                                                                                                 # infrastructure
process_parameters = (                                                                            # infrastructure
    width_curves[["true_order", "replication", "coefficients"]]                                  # infrastructure
    .drop_duplicates()                                                                             # infrastructure
    .sort_values(["true_order", "replication"])                                                   # infrastructure
    .reset_index(drop=True)                                                                        # infrastructure
)                                                                                                 # infrastructure
coefficient_source_hash = hashlib.sha256(                                                         # infrastructure
    process_parameters.to_csv(index=False).encode()                                                # infrastructure
).hexdigest()                                                                                      # infrastructure
population_settings = {                                                                           # infrastructure
    "method_version": 1,                                                                          # infrastructure
    "coefficient_source_hash": coefficient_source_hash,                                           # infrastructure
    "optimization_history": 3_000,                                                               # infrastructure
    "minimum_evaluation_history": 20_000,                                                        # infrastructure
    "maximum_evaluation_history": 250_000,                                                       # infrastructure
    "spectral_tail_target": 1e-10,                                                               # infrastructure
    "random_starts": 48,                                                                          # infrastructure
    "refined_starts": 8,                                                                          # infrastructure
}                                                                                                 # infrastructure
expected_population_rows = len(true_orders) * 50 * len(k_values)                                 # infrastructure
use_population_cache = (                                                                          # infrastructure
    population_width_path.is_file()                                                               # infrastructure
    and population_configuration_path.is_file()                                                   # infrastructure
    and json.loads(population_configuration_path.read_text()) == population_settings               # infrastructure
)                                                                                                 # infrastructure
if use_population_cache:                                                                          # infrastructure
    population_widths = pd.read_csv(population_width_path)                                        # infrastructure
    use_population_cache = (                                                                      # infrastructure
        len(population_widths) == expected_population_rows                                        # infrastructure
        and not population_widths.duplicated(["true_order", "replication", "k"]).any()            # infrastructure
        and np.isfinite(population_widths["population_minimum_width"]).all()                       # infrastructure
    )                                                                                              # infrastructure

if not use_population_cache:
    population_rows = []
    started = perf_counter()
    with threadpool_limits(limits=1):                                                              # infrastructure
        for process in process_parameters.itertuples(index=False):
            true_order = int(process.true_order)
            replication = int(process.replication)
            coefficients = np.asarray(json.loads(process.coefficients))
            approximation = approximate_population_width_curve(
                coefficients,
                seed=[20260922, true_order, replication],
            )
            for k, width in zip(k_values, approximation["widths"]):
                is_approximation = k <= true_order
                population_rows.append({
                    "true_order": true_order,
                    "replication": replication,
                    "k": int(k),
                    "population_minimum_width": float(width),
                    "method": (
                        "numerical approximation"
                        if is_approximation else "exact plateau formula"
                    ),
                    "history": (
                        approximation["evaluation_history"]
                        if is_approximation else 0
                    ),
                    "half_history_increment": (
                        float(approximation["half_history_increments"][k - 1])
                        if is_approximation else 0.0
                    ),
                    "plateau": approximation["plateau"],
                })
            if replication % 10 == 0:
                print(
                    f"Completed {replication}/50 AR({true_order}) "
                    "population curves"
                )
    population_widths = pd.DataFrame(population_rows)
    population_widths.to_csv(population_width_path, index=False)                                  # infrastructure
    population_configuration_path.write_text(                                                      # infrastructure
        json.dumps(population_settings, indent=2) + "\n"                                         # infrastructure
    )                                                                                              # infrastructure
    print(
        f"Generated 300 population curves in "
        f"{perf_counter() - started:.1f} seconds."
    )
else:
    print("Reused all 300 cached population-width curves.")

# Numerical quality checks.
population_groups = population_widths.groupby(["true_order", "replication"])
monotonicity_violations = sum(
    np.any(np.diff(group.sort_values("k")["population_minimum_width"]) > 1e-7)
    for _, group in population_groups
)
pre_plateau_violations = sum(
    np.any(
        group[group["k"] <= true_order]["population_minimum_width"]
        <= group["plateau"].iloc[0]
    )
    for (true_order, _), group in population_groups
)
relative_half_history_increment = (
    population_widths["half_history_increment"].abs()
    / population_widths["population_minimum_width"].clip(lower=1e-300)
)
assert monotonicity_violations == 0
assert pre_plateau_violations == 0
assert relative_half_history_increment.max() < 1.1e-5

# Repeat the hardest process at each order with an independent random-start seed.
radius_rows = []
for process in process_parameters.itertuples(index=False):
    coefficients = np.asarray(json.loads(process.coefficients))
    radius_rows.append({
        "true_order": int(process.true_order),
        "replication": int(process.replication),
        "coefficients": coefficients,
        "spectral_radius": float(max(abs(np.linalg.eigvals(companion_matrix(coefficients))))),
    })
radius_table = pd.DataFrame(radius_rows)
validation_differences = []
with threadpool_limits(limits=1):                                                                  # infrastructure
    for _, process in radius_table.sort_values(
        "spectral_radius", ascending=False
    ).groupby("true_order", sort=True).head(1).iterrows():
        repeated = approximate_population_width_curve(
            process["coefficients"],
            seed=[20260923, int(process["true_order"]), int(process["replication"])],
        )["widths"]
        cached = population_widths.loc[
            (population_widths["true_order"] == process["true_order"])
            & (population_widths["replication"] == process["replication"])
        ].sort_values("k")["population_minimum_width"].to_numpy()
        validation_differences.append(
            float(np.max(np.abs(repeated - cached) / np.maximum(cached, 1e-300)))
        )
assert max(validation_differences) < 1e-7

print(
    "Approximation checks: no monotonicity or plateau-bound violations; "
    f"maximum half-history increment {relative_half_history_increment.max():.2e}; "
    f"maximum independent-restart change {max(validation_differences):.2e}."
)


Reused all 300 cached population-width curves.
Approximation checks: no monotonicity or plateau-bound violations; maximum half-history increment 1.00e-05; maximum independent-restart change 5.73e-09.


### Population-width selection table

Each cell reports **correct / 50 (under / over)**.


In [11]:
population_selection_rows = []
for (true_order, replication), curve in population_widths.groupby(
    ["true_order", "replication"], sort=True
):
    minimum_widths = curve.sort_values("k")["population_minimum_width"].to_numpy()
    for spec in l_method_specs:
        fit = l_method_selector(
            minimum_widths,
            weighted=spec["weighted"],
            common_breakpoint=True,
            affine_affine=spec["affine_affine"],
            squared_loss=spec["squared_loss"],
        )
        population_selection_rows.append({
            "true_order": int(true_order),
            "replication": int(replication),
            "method": spec["method"],
            "selected_order": fit["selected_order"],
        })

population_selections = pd.DataFrame(population_selection_rows)
population_selections.to_csv(                                                                    # infrastructure
    RESULTS_DIR / "population_selections.csv", index=False                                       # infrastructure
)                                                                                                 # infrastructure
population_summary = (
    population_selections.assign(
        correct=lambda frame: frame["selected_order"] == frame["true_order"],
        under=lambda frame: frame["selected_order"] < frame["true_order"],
        over=lambda frame: frame["selected_order"] > frame["true_order"],
    )
    .groupby(["method", "true_order"], as_index=False)[["correct", "under", "over"]]
    .sum()
)
population_summary[["correct", "under", "over"]] = population_summary[
    ["correct", "under", "over"]
].astype(int)
population_summary["total"] = population_summary[["correct", "under", "over"]].sum(axis=1)
population_summary.to_csv(                                                                       # infrastructure
    RESULTS_DIR / "population_summary.csv", index=False                                          # infrastructure
)                                                                                                 # infrastructure
assert len(population_selections) == len(true_orders) * 50 * len(l_method_specs)
assert (population_summary["total"] == 50).all()

population_lookup = population_summary.set_index(["method", "true_order"])
population_index = pd.MultiIndex.from_tuples(
    [
        ("L-method", f"{spec['tail']} tail", spec["loss"], spec["score"])
        for spec in l_method_specs
    ],
    names=["Selector", "Tail", "Loss", "Score"],
)
population_table = pd.DataFrame(index=population_index)
for true_order in true_orders:
    entries = []
    for spec in l_method_specs:
        row = population_lookup.loc[(spec["method"], true_order)]
        entries.append(
            f"<strong>{int(row['correct'])}/50</strong><br>"
            f"<span>({int(row['under'])}/{int(row['over'])})</span>"
        )
    population_table[f"AR({true_order})"] = entries

population_table.to_csv(                                                                         # infrastructure
    RESULTS_DIR / "population_width_selector_table.csv"                                          # infrastructure
)                                                                                                 # infrastructure
styled_population_table = (
    population_table.style
    .set_caption("L-selectors applied to population minimum-width curves")
    .set_properties(**{
        "text-align": "center", "min-width": "78px",
        "padding": "8px 10px", "font-size": "14px",
    })
    .set_table_styles([
        {"selector": "table", "props": [
            ("border-collapse", "collapse"),
            ("font-family", "-apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif"),
            ("border", "1px solid #c7d0dc"),
        ]},
        {"selector": "caption", "props": [
            ("caption-side", "top"), ("font-size", "17px"),
            ("font-weight", "650"), ("color", "#17324d"),
            ("padding", "0 0 10px 0"), ("text-align", "left"),
        ]},
        {"selector": "thead th", "props": [
            ("background-color", "#17324d"), ("color", "white"),
            ("font-weight", "600"), ("padding", "8px 10px"),
            ("border", "1px solid #294b6d"),
        ]},
        {"selector": "tbody th", "props": [
            ("text-align", "left"), ("padding", "8px 10px"),
            ("background-color", "#f3f6f9"),
            ("border", "1px solid #d8dee7"), ("font-weight", "500"),
        ]},
        {"selector": "tbody td", "props": [
            ("border", "1px solid #d8dee7"), ("background-color", "white"),
        ]},
        {"selector": "tbody td span", "props": [
            ("color", "#5d6875"), ("font-size", "12px"),
        ]},
        {"selector": "tbody tr:nth-child(5) th, tbody tr:nth-child(5) td", "props": [
            ("border-top", "2px solid #aebdca"),
        ]},
        {"selector": "tbody tr:hover td", "props": [
            ("background-color", "#eef5fb"),
        ]},
    ])
)
styled_population_table


In [12]:
empirical_l_selections = selections[selections["method"].str.startswith("L —")]
attribution = empirical_l_selections.merge(
    population_selections,
    on=["method", "true_order", "replication"],
    suffixes=("_empirical", "_population"),
)
attribution["empirical_correct"] = (
    attribution["selected_order_empirical"] == attribution["true_order"]
)
attribution["population_correct"] = (
    attribution["selected_order_population"] == attribution["true_order"]
)

attribution_rows = []
for spec in l_method_specs:
    group = attribution[attribution["method"] == spec["method"]]
    attribution_rows.append({
        "Empirical correct": f"{int(group['empirical_correct'].sum())}/300",
        "Population correct": f"{int(group['population_correct'].sum())}/300",
        "Same selected order": f"{int((group['selected_order_empirical'] == group['selected_order_population']).sum())}/300",
        "Wrong in both": f"{int((~group['empirical_correct'] & ~group['population_correct']).sum())}/300",
    })
attribution_table = pd.DataFrame(attribution_rows, index=population_index)
attribution_table.to_csv(                                                                        # infrastructure
    RESULTS_DIR / "empirical_population_attribution.csv"                                         # infrastructure
)                                                                                                 # infrastructure

styled_attribution_table = (
    attribution_table.style
    .set_caption("Empirical versus population-width diagnosis")
    .set_properties(**{
        "text-align": "center", "padding": "8px 10px", "font-size": "14px",
    })
    .set_table_styles([
        {"selector": "table", "props": [
            ("border-collapse", "collapse"),
            ("font-family", "-apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif"),
            ("border", "1px solid #c7d0dc"),
        ]},
        {"selector": "caption", "props": [
            ("caption-side", "top"), ("font-size", "17px"),
            ("font-weight", "650"), ("color", "#17324d"),
            ("padding", "14px 0 10px 0"), ("text-align", "left"),
        ]},
        {"selector": "thead th", "props": [
            ("background-color", "#17324d"), ("color", "white"),
            ("font-weight", "600"), ("padding", "8px 10px"),
            ("border", "1px solid #294b6d"),
        ]},
        {"selector": "tbody th", "props": [
            ("text-align", "left"), ("padding", "8px 10px"),
            ("background-color", "#f3f6f9"),
            ("border", "1px solid #d8dee7"), ("font-weight", "500"),
        ]},
        {"selector": "tbody td", "props": [
            ("border", "1px solid #d8dee7"), ("background-color", "white"),
        ]},
        {"selector": "tbody tr:nth-child(5) th, tbody tr:nth-child(5) td", "props": [
            ("border-top", "2px solid #aebdca"),
        ]},
    ])
)
styled_attribution_table


### Diagnosis

The L-selector is the principal failure point. Its strongest fixed variant—affine head, constant tail, squared loss, and weighted RMSE—selects **163/300** orders correctly from the population curves, essentially the same as its **164/300** empirical result. For that variant, the empirical and population selectors choose the same order in **247/300** processes and are both wrong in **123/300**. Replacing the estimated curves by population minima therefore does not repair the method.

Every population-curve error is an **underselection**. The population curves satisfy the theoretically required strict decrease through $k=p$ and exact plateau from $k=p+1$, yet the pre-plateau part need not resemble one affine line. In near-boundary processes, its earliest widths can be orders of magnitude above the plateau. The global two-segment fit then minimizes residual loss by explaining an earlier, larger bend and ignores the final transition into the plateau. This is an inference from the population table and the computed curves, consistent with the population-width structure in [Proposition `prop:minimum_width`](../../Abschlussarbeit_2021/LaTeX/chapters/3_application_to_time_series.tex).

This diagnostic does **not** prove that every projected-subgradient run found the exact empirical minimum. It shows something sufficient for the present conclusion: even supplying the L-selector with population minimum-width curves leaves its accuracy essentially unchanged. Numerical minimizer failure is therefore not the main explanation for the poor order recovery.


## Analytic counterexample: definite selector failure

The simulation tables are strong numerical evidence, but by themselves they are not a mathematical proof. The following causal AR(2) process gives an analytic counterexample that does not require numerically approximating $m_2$.

Let
$$
(1-aB)(1-bB)X_t=Z_t,qquad a=0.99,quad b=0.1,quad Z_t\sim\operatorname{Unif}[-1,1].
$$
This is a genuine causal AR(2) process with $(\phi_1,\phi_2)=(a+b,-ab)=(1.09,-0.099)$. Write $M=m_1$, $x=m_2$, and $c=m_3=\cdots=m_{10}$. Since the impulse-response coefficients are positive,
$$
M=\frac{2}{(1-a)(1-b)}=222.222222\ldots,
$$
while the exact plateau formula gives
$$
c=\frac{2}{\sqrt{1+(a+b)^2+(ab)^2}}=1.349043\ldots.
$$
The population theorem gives $x>c$. Evaluating the population width in the normalized direction $(1,-a)/\sqrt{1+a^2}$ cancels the factor $1-aB$ and gives the upper bound
$$
x\le d:=\frac{2}{(1-b)\sqrt{1+a^2}}=1.579225\ldots.
$$
Consequently, the unknown exact value obeys $c<x\le d$.

Consider any of the shared-breakpoint selectors with $K=10$. At candidate order $q=1$, the affine head interpolates $(M,x)$ exactly. A constant tail fitted to $(x,c,\ldots,c)$ has
$$
\operatorname{SSE}_{q=1}=\frac89(x-c)^2\le\frac89(d-c)^2=0.047097\ldots,
\qquad
\operatorname{SAE}_{q=1}=x-c\le d-c=0.230181\ldots.
$$
An affine tail can only reduce these losses, so the same upper bounds apply to both tail models.

For every $q\ge2$, the affine head contains the first three values $(M,x,c)$. The least-squares residual on these three points is $(M-2x+c)^2/6$, and the least-absolute residual is $|M-2x+c|/2$. Adding further head observations cannot reduce either residual on the first three points. Since $x\le d$,
$$
\operatorname{SSE}_{q\ge2}\ge\frac{(M-2d+c)^2}{6}=8096.968264\ldots,
\qquad
\operatorname{SAE}_{q\ge2}\ge\frac{M-2d+c}{2}=110.206408\ldots.
$$
Thus $q=1$ strictly beats every $q\ge2$ for total squared and total absolute loss. For the weighted scores, the common-breakpoint segment counts always sum to $K+1=11$, and the corresponding bounds are
$$
S_{\mathrm{WRMSE}}(1)\le\frac{\sqrt8}{11}(d-c)=0.059187\ldots
<\frac{M-2d+c}{11\sqrt2}=14.168672\ldots\le S_{\mathrm{WRMSE}}(q),
$$
and
$$
S_{\mathrm{WMAE}}(1)\le\frac{d-c}{11}=0.020926\ldots
<\frac{M-2d+c}{22}=10.018764\ldots\le S_{\mathrm{WMAE}}(q).
$$

Therefore **all eight L variants select $\widehat p=1$ although the true order is $p=2$**, even when supplied with the exact population minimum-width curve. This proves that the proposed L-selector is not guaranteed to recover the population breakpoint. It does not prove that the empirical width optimizer is always accurate; it proves that optimizer accuracy cannot fix this structural selector failure. The population-width facts used above are those of [Proposition `prop:minimum_width`](../../Abschlussarbeit_2021/LaTeX/chapters/3_application_to_time_series.tex), and the scores are exactly those implemented in [`l_method_selector`](ar_tools/L_method.py).
